# Chronos-2 trading features with native REINFORCE

This notebook shows the two Chronos workflows supported by `crosslearn`:

1. online extraction with `ChronosExtractor`
2. offline precomputation with `ChronosEmbedder`

`gym-anytrading` stays outside the package extras on purpose, so install it directly at the notebook level.
            

In [ ]:
# Uncomment in a fresh environment:
# %pip install "crosslearn[chronos]" gym-anytrading pandas matplotlib      

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from gym_anytrading.datasets import STOCKS_GOOGL
from gym_anytrading.envs import StocksEnv

from crosslearn import REINFORCE, make_vec_env
from crosslearn.extractors import ChronosEmbedder, ChronosExtractor        

## Dataset and rolling-window setup

The raw environment emits rolling OHLCV windows. `ChronosExtractor` can embed those windows online, while `ChronosEmbedder` can precompute the same representation into dataframe columns ahead of training.
            

In [ ]:
LOOKBACK = 30
FEATURE_COLUMNS = ['Open', 'High', 'Low', 'Close', 'Volume']
SELECTED_COLUMNS = ['Close', 'Volume']
FRAME_BOUND = (LOOKBACK, len(STOCKS_GOOGL))

print(STOCKS_GOOGL.head())
print('Frame bound:', FRAME_BOUND)
print('Feature columns:', FEATURE_COLUMNS)
print('Chronos-selected columns:', SELECTED_COLUMNS) 

In [ ]:
def online_process_data(env):
    start = env.frame_bound[0] - env.window_size
    end = env.frame_bound[1]
    prices = env.df.loc[:, 'Close'].to_numpy()[start:end]
    signal_features = env.df.loc[:, FEATURE_COLUMNS].to_numpy(dtype=np.float32)[start:end]
    return prices, signal_features


class OnlineStocksEnv(StocksEnv):
    _process_data = online_process_data


def make_online_env():
    return OnlineStocksEnv(
        df=STOCKS_GOOGL,
        window_size=LOOKBACK,
        frame_bound=FRAME_BOUND,
    )

sample_env = make_online_env()
print('Online observation space:', sample_env.observation_space)
sample_env.close()

### Train without Chronos extractor
First, let's train an agent without Chronos to establish a baseline performance. This agent will use the raw OHLCV windows as input.

In [ ]:
def render_single_episode(env, agent):
    obs, _ = env.reset()
    terminated = False
    truncated = False
    
    while not (terminated or truncated):
        action = agent.predict(obs, deterministic=True)
        obs, reward, terminated, truncated, info = env.step(action)
    
    print(f"Info: {info}")
    env.unwrapped.render_all()
    plt.show()
    env.close()

In [ ]:
vec_base = make_vec_env(make_online_env, n_envs=1)

online_base_agent = REINFORCE(
    vec_base,
    n_steps=512,
    seed=42,
    verbose=2,
)
online_base_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_online_env(), online_base_agent)

## Online Chronos extraction

The environment still emits raw windows. `ChronosExtractor` performs the embedding step inside the policy so the agent can train directly on the encoded representation. This notebook keeps `n_envs=1` for clarity; on GPU, raise `n_envs` to build larger Chronos batches and use `use_async_env=True` only when environment stepping becomes the bottleneck.
            

In [ ]:
vec_env = make_vec_env(make_online_env, n_envs=1)

online_agent = REINFORCE(
    vec_env,
    n_steps=512,
    features_extractor_class=ChronosExtractor,
    features_extractor_kwargs={
        'feature_names': FEATURE_COLUMNS,
        'selected_columns': SELECTED_COLUMNS,
    },
    seed=42,
    verbose=2,
)
online_agent.learn(total_timesteps=10_000)

In [ ]:
render_single_episode(make_online_env(), online_agent)

## Offline Chronos embeddings

For offline mode, the rolling windows are embedded once, appended as `chronos_*` columns, and then used as the observation features for a second environment. Set `progress_bar=True` to monitor the offline embedding pass.
            

In [ ]:
def build_offline_bundle(df, lookback, frame_bound):
    start = frame_bound[0] - lookback
    end = frame_bound[1]
    history = df.iloc[start:end].reset_index(drop=True).copy()

    embedder = ChronosEmbedder(
        feature_names=FEATURE_COLUMNS,
        selected_columns=SELECTED_COLUMNS,
    )
    embedded = embedder.transform_dataframe(
        history,
        lookback=lookback,
        columns=FEATURE_COLUMNS,
        progress_bar=True,
    )
    embedding_columns = [
        column for column in embedded.columns if column.startswith('chronos_')
    ]
    embedded = embedded.iloc[lookback - 1 :].reset_index(drop=True)

    return {
        'df': embedded,
        'prices': embedded.loc[:, 'Close'].to_numpy(dtype=np.float32),
        'signal_features': embedded.loc[:, embedding_columns].to_numpy(dtype=np.float32),
        'embedding_columns': embedding_columns,
    }


offline_bundle = build_offline_bundle(STOCKS_GOOGL, LOOKBACK, FRAME_BOUND)
print(offline_bundle['df'].head())
print('Offline feature matrix shape:', offline_bundle['signal_features'].shape)
print('Offline embedding columns:', offline_bundle['embedding_columns'][:5], '...')     

In [ ]:
class OfflineStocksEnv(StocksEnv):
    def __init__(self, prices, signal_features, **kwargs):
        self._prices = prices
        self._signal_features = signal_features.astype(np.float32)
        super().__init__(**kwargs)

    def _process_data(self):
        return self._prices, self._signal_features


def make_offline_env():
    return OfflineStocksEnv(
        prices=offline_bundle['prices'],
        signal_features=offline_bundle['signal_features'],
        df=offline_bundle['df'],
        window_size=1,
        frame_bound=(1, len(offline_bundle['df'])),
    )

vec_offline = make_vec_env(make_offline_env, n_envs=1)
offline_agent = REINFORCE(vec_offline, n_steps=512, seed=42, verbose=2)
offline_agent.learn(total_timesteps=10_000)         

In [ ]:
render_single_episode(make_offline_env(), offline_agent)